In [0]:
import pandas as pd
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

BASE_PATH = "/Workspace/Users/prarthanabg7@gmail.com/E-commerce/01_Data/"

print("Spark ready")

Spark ready


In [0]:
orders_pd    = pd.read_csv(BASE_PATH + "olist_orders_dataset.csv")
payments_pd  = pd.read_csv(BASE_PATH + "olist_order_payments_dataset.csv")
products_pd  = pd.read_csv(BASE_PATH + "olist_products_dataset.csv")
customers_pd = pd.read_csv(BASE_PATH + "olist_customers_dataset.csv")
items_pd     = pd.read_csv(BASE_PATH + "olist_order_items_dataset.csv")
reviews_pd   = pd.read_csv(BASE_PATH + "olist_order_reviews_dataset.csv")
category_pd  = pd.read_csv(BASE_PATH + "product_category_name_translation.csv")

print("All files loaded")

All files loaded


In [0]:
#Verify Shape and Columns
datasets = {
    "orders": orders_pd,
    "payments": payments_pd,
    "products": products_pd,
    "customers": customers_pd,
    "items": items_pd,
    "reviews": reviews_pd,
    "category": category_pd
}

for name, df in datasets.items():
    print(f"{name}: {df.shape} | columns: {df.columns.tolist()}")

orders: (99441, 8) | columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
payments: (103886, 5) | columns: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
products: (32951, 9) | columns: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
customers: (99441, 5) | columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
items: (112650, 7) | columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
reviews: (99224, 7) | columns: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_

In [0]:
print("=== NULL COUNTS ===\n")
for name, df in datasets.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) > 0:
        print(f"--- {name} ---")
        print(nulls)
        print()
    else:
        print(f"--- {name} --- ✅ No nulls\n")

=== NULL COUNTS ===

--- orders ---
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

--- payments --- ✅ No nulls

--- products ---
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

--- customers --- ✅ No nulls

--- items --- ✅ No nulls

--- reviews ---
review_comment_title      87656
review_comment_message    58247
dtype: int64

--- category --- ✅ No nulls



In [0]:
print("=== DUPLICATE COUNTS ===\n")

pk_map = {
    "orders":    "order_id",
    "payments":  "order_id",
    "products":  "product_id",
    "customers": "customer_id",
    "items":     "order_id",
    "reviews":   "order_id",
    "category":  "product_category_name"
}

for name, pk in pk_map.items():
    df = datasets[name]
    dupes = df.duplicated(subset=[pk]).sum()
    status = "✅" if dupes == 0 else "⚠️"
    print(f"{status} {name:12} → duplicates on '{pk}': {dupes}")

=== DUPLICATE COUNTS ===

✅ orders       → duplicates on 'order_id': 0
⚠️ payments     → duplicates on 'order_id': 4446
✅ products     → duplicates on 'product_id': 0
✅ customers    → duplicates on 'customer_id': 0
⚠️ items        → duplicates on 'order_id': 13984
⚠️ reviews      → duplicates on 'order_id': 551
✅ category     → duplicates on 'product_category_name': 0
